# Build the analysis-ready vacancy dataset

This notebook reads the **monthly full Parquet files** produced by Stage 5 of the data pipeline, retains Ukrainian vacancies, and collapses repeated observations to one row per vacancy identifier.

The workflow is memory intensive. Run it manually in an environment with sufficient RAM. It is intentionally not executed or stored with outputs in the replication package.

## Inputs and outputs

Input (created by Stage 5):

- `data/data-pipeline/stage_05/parquet_monthly_full/<year>-<month>.parquet`

Outputs:

- yearly intermediate files in `data/paper-analytics/interim/`
- final analysis-ready file: `data/paper-analytics/analysis-ready/vacancies_2021_2025_collapsed_by_id.parquet`

All paths are repository-relative. The code locates the repository root from the current working directory; no machine-specific path is required.

In [ ]:
import gc
from pathlib import Path

import pandas as pd

## Configuration

The analysis covers monthly files dated from 2021 through 2025. The code processes every correctly named monthly file available for those years and reports the exact files used.

In [ ]:
def find_repository_root(start: Path | None = None) -> Path:
    """Find the repository root from a working directory inside the repository."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "data-pipeline" / "stage_05").is_dir():
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Start Jupyter from this replication repository."
    )


REPOSITORY_ROOT = find_repository_root()
INPUT_DIR = (
    REPOSITORY_ROOT
    / "data"
    / "data-pipeline"
    / "stage_05"
    / "parquet_monthly_full"
)
INTERIM_DIR = REPOSITORY_ROOT / "data" / "paper-analytics" / "interim"
OUTPUT_DIR = REPOSITORY_ROOT / "data" / "paper-analytics" / "analysis-ready"
FINAL_FILE = OUTPUT_DIR / "vacancies_2021_2025_collapsed_by_id.parquet"
YEARS = tuple(range(2021, 2026))
COUNTRY = "Ukraine"

if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Stage 5 input directory does not exist: {INPUT_DIR}")

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## File discovery and validation

Monthly files must follow the Stage 5 naming convention `<year>-<month>.parquet`. Files outside 2021–2025 are ignored. At least one monthly file must exist for every included year.

In [ ]:
def monthly_file_sort_key(path: Path) -> tuple[int, int]:
    """Return the numeric year and month encoded in a Stage 5 filename."""
    try:
        year_text, month_text = path.stem.split("-", maxsplit=1)
        year, month = int(year_text), int(month_text)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"Unexpected monthly filename: {path.name}") from exc

    if month not in range(1, 13):
        raise ValueError(f"Invalid month in filename: {path.name}")
    return year, month


monthly_files = sorted(INPUT_DIR.glob("*.parquet"), key=monthly_file_sort_key)
monthly_files = [path for path in monthly_files if monthly_file_sort_key(path)[0] in YEARS]

if not monthly_files:
    raise FileNotFoundError(f"No monthly Parquet files found in {INPUT_DIR}")

files_by_year = {
    year: [path for path in monthly_files if monthly_file_sort_key(path)[0] == year]
    for year in YEARS
}
missing_years = [year for year, paths in files_by_year.items() if not paths]
if missing_years:
    raise FileNotFoundError(f"No monthly files found for years: {missing_years}")

for year, paths in files_by_year.items():
    print(f"{year}: {len(paths)} file(s): {', '.join(path.name for path in paths)}")

## Collapse rules

The same aggregation rules are applied when monthly files are joined within a year and when yearly files are joined into the final dataset. For each vacancy `id`, dates retain the earliest observed value, clicks retain their maximum and mean, and descriptive vacancy fields retain the first available value.

Object-valued fields are stored as nullable strings in the intermediate Parquet files. This keeps Parquet schemas stable while preserving missing values.

In [ ]:
REQUIRED_SOURCE_COLUMNS = {
    "id",
    "date_created",
    "number_of_clicks",
    "date",
    "clean_title",
    "clean_desc",
    "min_salary",
    "max_salary",
    "currency",
    "salary_rate",
    "date_expired",
    "esco_title",
    "esco_skills",
    "region",
    "region_original",
    "city",
    "country",
    "latitude",
    "longitude",
    "classified_code",
    "classified_title",
    "desc_lang",
    "skill_labels",
    "skill_labels_en",
}

SOURCE_AGGREGATIONS = {
    "date_created": "first",
    "date_created_dateformat2": "first",
    "number_of_clicks": ["max", "mean"],
    "date": "min",
    "clean_title": "first",
    "clean_desc": "first",
    "min_salary": "first",
    "max_salary": "first",
    "currency": "first",
    "salary_rate": "first",
    "date_expired": "first",
    "esco_title": "first",
    "esco_skills": "first",
    "region": "first",
    "region_original": "first",
    "city": "first",
    "country": "first",
    "latitude": "first",
    "longitude": "first",
    "classified_code": "first",
    "classified_title": "first",
    "desc_lang": "first",
    "skill_labels": "first",
    "skill_labels_en": "first",
}

COLLAPSED_AGGREGATIONS = {
    "date_created": "first",
    "date_created_dateformat2": "first",
    "number_of_clicks_max": "max",
    "number_of_clicks_mean": "mean",
    "date": "min",
    "clean_title": "first",
    "clean_desc": "first",
    "min_salary": "first",
    "max_salary": "first",
    "currency": "first",
    "salary_rate": "first",
    "date_expired": "first",
    "esco_title": "first",
    "esco_skills": "first",
    "region": "first",
    "region_original": "first",
    "city": "first",
    "country": "first",
    "latitude": "first",
    "longitude": "first",
    "classified_code": "first",
    "classified_title": "first",
    "desc_lang": "first",
    "skill_labels": "first",
    "skill_labels_en": "first",
}

In [ ]:
def validate_source_columns(data: pd.DataFrame, source_file: Path) -> None:
    """Fail early when a Stage 5 file does not match the expected schema."""
    missing = sorted(REQUIRED_SOURCE_COLUMNS.difference(data.columns))
    if missing:
        raise KeyError(f"{source_file.name} is missing required columns: {missing}")


def flatten_aggregation_columns(data: pd.DataFrame) -> pd.DataFrame:
    """Flatten columns produced by multiple click aggregations."""
    data.columns = [
        "_".join(part for part in column if part).rstrip("_")
        if isinstance(column, tuple)
        else column
        for column in data.columns
    ]
    return data


def restore_creation_date(data: pd.DataFrame) -> pd.DataFrame:
    """Use the earliest observation date when the recorded creation date equals zero."""
    zero_creation_date = pd.to_numeric(data["date_created"], errors="coerce").eq(0)
    data.loc[zero_creation_date, "date_created_dateformat2"] = pd.to_datetime(
        data.loc[zero_creation_date, "date"], errors="coerce"
    )
    data["year_created"] = data["date_created_dateformat2"].dt.year.astype("Int64")
    return data


def make_parquet_schema_stable(data: pd.DataFrame) -> pd.DataFrame:
    """Convert object-valued fields to nullable strings before writing Parquet."""
    for column in data.select_dtypes(include="object").columns:
        data[column] = data[column].astype("string")
    return data

## Step 1: collapse monthly observations within each year

Each monthly file is loaded separately and immediately filtered to Ukrainian vacancies. The year is then collapsed and written to disk before the next year is processed, limiting peak memory usage. Any unreadable or structurally invalid input stops the workflow instead of producing a silently incomplete result.

In [ ]:
yearly_files = []

for year, source_files in files_by_year.items():
    monthly_frames = []

    for source_file in source_files:
        print(f"Reading {source_file.name}")
        monthly_data = pd.read_parquet(source_file)
        validate_source_columns(monthly_data, source_file)

        # Stage 5 monthly-full files contain all countries. This paper uses Ukraine only.
        monthly_data = monthly_data.loc[monthly_data["country"].eq(COUNTRY)].copy()
        monthly_data["date_created_dateformat2"] = pd.to_datetime(
            monthly_data["date_created"], errors="coerce"
        )

        collapsed_month = (
            monthly_data.groupby("id", as_index=False, sort=False)
            .agg(SOURCE_AGGREGATIONS)
            .pipe(flatten_aggregation_columns)
        )
        monthly_frames.append(collapsed_month)

        del monthly_data, collapsed_month
        gc.collect()

    year_data = pd.concat(monthly_frames, ignore_index=True)
    if year_data.empty:
        raise ValueError(f"No {COUNTRY} observations found in the {year} monthly files.")
    del monthly_frames
    gc.collect()

    year_data = (
        year_data.groupby("id", as_index=False, sort=False)
        .agg(COLLAPSED_AGGREGATIONS)
        .pipe(restore_creation_date)
        .pipe(make_parquet_schema_stable)
    )

    yearly_file = INTERIM_DIR / f"vacancies_{year}_collapsed_by_id.parquet"
    year_data.to_parquet(yearly_file, index=False)
    yearly_files.append(yearly_file)
    print(f"Saved {len(year_data):,} rows to {yearly_file}")

    del year_data
    gc.collect()

## Step 2: combine yearly files and remove cross-year duplicates

Vacancies may appear in more than one yearly file. The final aggregation applies the same rules once more so that every `id` occurs exactly once.

In [ ]:
yearly_frames = [pd.read_parquet(path) for path in yearly_files]
combined_data = pd.concat(yearly_frames, ignore_index=True)
del yearly_frames
gc.collect()

final_data = (
    combined_data.groupby("id", as_index=False, sort=False)
    .agg(COLLAPSED_AGGREGATIONS)
    .pipe(restore_creation_date)
    .pipe(make_parquet_schema_stable)
)
del combined_data
gc.collect()

if final_data["id"].duplicated().any():
    raise AssertionError("The final dataset contains duplicate vacancy identifiers.")

final_data.to_parquet(FINAL_FILE, index=False)
print(f"Saved {len(final_data):,} unique vacancies to {FINAL_FILE}")

## Step 3: lightweight output checks

These checks inspect the in-memory result only. They do not reload the full output file.

In [ ]:
summary = pd.Series(
    {
        "rows": len(final_data),
        "unique_ids": final_data["id"].nunique(dropna=False),
        "minimum_observation_date": final_data["date"].min(),
        "maximum_observation_date": final_data["date"].max(),
        "missing_creation_dates": final_data["date_created_dateformat2"].isna().sum(),
    },
    name="value",
)
summary